In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("EcommerceCaseStudy").getOrCreate()
customers = spark.read.option("header",True).csv("data/customers.csv", inferSchema=True)
products = spark.read.option("header",True).csv("data/products.csv", inferSchema=True)
orders = spark.read.option("header",True).csv("data/orders.csv", inferSchema=True)
order_items = spark.read.option("header",True).csv("data/order_items.csv", inferSchema=True)
returns = spark.read.option("header",True).csv("data/returns.csv", inferSchema=True)

In [5]:
# Q1
print(customers.count())
print(products.count())
print(orders.count())
print(returns.select("order_id").distinct().count())

100000
50000
1000000


[Stage 19:>                                                         (0 + 1) / 1]

100000


In [ ]:
# Q2
q2 = order_items.join(products,"product_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("category").agg(sum("revenue").alias("total_sales"))

In [7]:
# Q3
q3 = customers.join(orders,"customer_id") \
.join(order_items,"order_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("customer_id","customer_name") \
.agg(sum("revenue").alias("total_spend")) \
.orderBy(desc("total_spend")).limit(10)

q3.show()

[Stage 30:=============================>                            (1 + 1) / 2]

+-----------+--------------+------------------+
|customer_id| customer_name|       total_spend|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|169060.39999999997|
|      23289|Customer_23289|161573.80000000002|
|      52275|Customer_52275|153364.78999999998|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|         152680.05|
|      40442|Customer_40442|         151037.32|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|148281.03999999998|
+-----------+--------------+------------------+



In [ ]:
# Q4
q4 = orders.join(order_items,"order_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.withColumn("year", year("order_date")) \
.withColumn("month", month("order_date")) \
.groupBy("year","month") \
.agg(sum("revenue").alias("monthly_sales")) \
.orderBy("year","month")

In [ ]:
# Q5
total_orders = products.join(order_items,"product_id") \
.groupBy("category").agg(countDistinct("order_id").alias("total_orders"))
returned_orders = returns.join(order_items,"order_id") \
.join(products,"product_id") \
.groupBy("category") \
.agg(countDistinct("order_id").alias("returned_orders"))
q5 = total_orders.join(returned_orders,"category","left") \
.fillna(0) \
.withColumn("return_pct",(col("returned_orders")/col("total_orders"))*100)

In [ ]:
# Q6
state_payment = customers.join(orders,"customer_id") \
.groupBy("state","payment_mode").count()
w = Window.partitionBy("state").orderBy(desc("count"))
q6 = state_payment.withColumn("rank",row_number().over(w)).filter("rank=1")

In [ ]:
# Q7
q7 = customers.join(orders,"customer_id") \
.join(order_items,"order_id") \
.join(products,"product_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("customer_id","customer_name") \
.agg(countDistinct("category").alias("categories"),
     sum("revenue").alias("spend")) \
.filter((col("categories") >= 5) & (col("spend") > 100000))

In [ ]:
# Q8
product_rev = products.join(order_items,"product_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("category","product_id","product_name") \
.agg(sum("revenue").alias("revenue"))
w = Window.partitionBy("category").orderBy(desc("revenue"))
q8 = product_rev.withColumn("rank", dense_rank().over(w)).filter("rank<=3")

In [ ]:
# Q9
q9 = customers.join(orders,"customer_id") \
.join(order_items,"order_id") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("customer_id","customer_name") \
.agg(sum("revenue").alias("clv")) \
.withColumn("segment",
    when(col("clv") < 50000,"Bronze")
    .when(col("clv") <= 150000,"Silver")
    .otherwise("Gold"))

In [ ]:
# Q10
final_df = customers.join(orders,"customer_id","left") \
.join(order_items,"order_id","left") \
.join(returns,"order_id","left") \
.withColumn("revenue", col("quantity")*col("selling_price")) \
.groupBy("customer_id","customer_name","state","customer_segment") \
.agg(countDistinct("order_id").alias("total_orders"),
     sum("revenue").alias("total_revenue"),
     countDistinct(returns.return_id).alias("total_returns"))
final_df.write.mode("overwrite").partitionBy("state").parquet("final_output")